# Retail Sales - Data Cleaning & Feature Engineering 

i/o : "C:\Users\shahul\Downloads\royal_store_raw_data.csv"
o/p : "C:\Users\shahul\Downloads\royal_store_cleaned"

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

## 1. Load Data

In [7]:
df = pd.read_csv(r"C:\Users\shahul\Downloads\royal_store_raw_data.csv")
print(f"Shape : {df.shape}")
print(f"Columns : {list(df.columns)}")
df.head(5)

Shape : (131265, 10)
Columns : ['Bill.NO', 'Date', 'Product Name', 'Qty', 'Rate', 'Amount', 'Tax%', 'Sheet_Name', 'Total', 'IsReturn']


,Bill.NO,Date,Product Name,Qty,Rate,Amount,Tax%,Sheet_Name,Total,IsReturn
0,1,2025-04-01,GOLDWINNER 200ML,1.0,35.0,35.0,0.0,Sheet1,35.0,False
1,1,2025-04-01,BRU GREEN LABEL 100G,-1.0,78.0,-78.0,5.0,Sheet3,-78.0,True
2,2,2025-04-01,CARRY BAG 13*16 WHITE PAK,1.0,35.0,35.0,0.0,Sheet1,35.0,False
3,2,2025-04-01,JOHNSONS BABY POWDER BLOSSMS 100G,-1.0,125.0,-125.0,18.0,Sheet3,-125.0,True
4,3,2025-04-01,GULAS JAGGERY 500GM,1.0,47.0,47.0,0.0,Sheet1,47.0,False


In [9]:
df.info()

print(df.isnull().sum())

df.describe(include='all')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 131265 entries, 0 to 131264
Data columns (total 10 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Bill.NO       131265 non-null  int64  
 1   Date          131265 non-null  object 
 2   Product Name  131265 non-null  object 
 3   Qty           131191 non-null  float64
 4   Rate          131265 non-null  float64
 5   Amount        131191 non-null  float64
 6   Tax%          131265 non-null  float64
 7   Sheet_Name    131265 non-null  object 
 8   Total         131191 non-null  float64
 9   IsReturn      131265 non-null  bool   
dtypes: bool(1), float64(5), int64(1), object(3)
memory usage: 9.1+ MB
Bill.NO          0
Date             0
Product Name     0
Qty             74
Rate             0
Amount          74
Tax%             0
Sheet_Name       0
Total           74
IsReturn         0
dtype: int64


,Bill.NO,Date,Product Name,Qty,Rate,Amount,Tax%,Sheet_Name,Total,IsReturn
count,131265.000000,131265,131265,131191.000000,131265.000000,131191.000000,131265.000000,131265,131191.000000,131265
unique,NaN,99,4616,NaN,NaN,NaN,NaN,3,NaN,2
top,NaN,2025-04-06,IDLY MAVU,NaN,NaN,NaN,NaN,Sheet1,NaN,False
freq,NaN,2416,3279,NaN,NaN,NaN,NaN,56093,NaN,131142
mean,19554.496964,NaN,NaN,2.010957,50.067825,60.422542,4.209058,NaN,60.422542,NaN
std,11537.136468,NaN,NaN,33.734477,79.360654,107.019463,7.334880,NaN,107.019463,NaN
min,1.000000,NaN,NaN,-25.000000,1.000000,-1700.000000,0.000000,NaN,-1700.000000,NaN
25%,9459.000000,NaN,NaN,1.000000,14.000000,20.000000,0.000000,NaN,20.000000,NaN
50%,19645.000000,NaN,NaN,1.000000,30.000000,38.000000,0.000000,NaN,38.000000,NaN
75%,29489.000000,NaN,NaN,1.000000,55.000000,68.000000,5.000000,NaN,68.000000,NaN


# 2. Remove Non-Product / Junk Entries

In [10]:
# Products with Rate <=1 are service entries, not real products
print("Products with Rate == 1:")
print(df[df["Rate"] == 1]["Product Name"].value_counts().head(10))

Products with Rate == 1:
Product Name
SILLARAI RS 1.00                  401
CANDYMAN MRP 1.00                 299
SUNSILK MRP 1                     173
KADALAMITTAI RS.1                 131
CHIK SHAMPOO JASMINE RS 1/         65
SHAMPOO  RS 1/  EGG WHITE          52
BILL AMOUNT                        35
KARTHIKA SHAMPOO 20% RS.1/-        21
CLINIC PLAS +HIBISCUS MRP 1.00      9
CLINIC PLUS EGG MRP 1               2
Name: count, dtype: int64

After removing junk: (130829, 10)


In [11]:
# Remove non-product entries
df = df[~df["Product Name"].isin(["SILLARAI RS 1.00", "BILL AMOUNT"])]
print(f"\nAfter removing junk: {df.shape}")


After removing junk: (130829, 10)


# 3. Validate Amount & Drop Nulls

In [12]:
# Verify Amount = Qty × Rate
df['Calc_amt'] = df["Qty"] * df["Rate"]
mismatch = df[abs(df["Amount"] - df['Calc_amt']) > 0.01]
print(f"Amount mismatches : {len(mismatch)}")

Amount mismatches : 0


In [13]:
# Drop rows with nulls in key columns
cols_with_nan = ['Qty', 'Amount', 'Total', 'Calc_amt']
before = len(df)
df = df.dropna(subset=cols_with_nan)
print(f"Rows dropped      : {before - len(df)}")
print(f"Rows remaining    : {len(df)}")

Rows dropped      : 73
Rows remaining    : 130756


# 4. Feature Engineering

In [ ]:
# Sales category from Qty
def classify_qty(qty):
    if qty < 0:      return 'Returns/adjustments'
    elif qty == 1:   return 'Retail sales'
    elif qty <= 5:   return 'Small multi-buy'
    elif qty <= 20:  return 'Medium purchase'
    else:            return 'Bulk sales (valid)'

df['Sales_Category'] = df['Qty'].apply(classify_qty)

In [ ]:
# Date features
df['Date']    = pd.to_datetime(df['Date'], errors='coerce')
df['Year']    = df['Date'].dt.year
df['Month']   = df['Date'].dt.month
df['Day']     = df['Date'].dt.day
df['Weekday'] = df['Date'].dt.day_name()

In [ ]:
# Round numeric columns
for col in ['Total', 'Amount', 'Rate']:
    df[col] = df[col].round(2)

# Add IsReturn flag
df['IsReturn'] = df['Qty'] < 0

print("Features added!")
print(df[['Date','Year','Month','Weekday','Sales_Category','IsReturn']].head())

# 5. Remove Duplicates & Drop Unused Columns

In [14]:
# Duplicates
before = len(df)
df.drop_duplicates(subset=['Bill.NO', 'Product Name'], keep='first', inplace=True)
print(f"Duplicates removed : {before - len(df)}")

# Drop columns not needed for analysis
drop_cols = [c for c in ['Sheet_Name', 'Calc_amt', 'Tax%'] if c in df.columns]
df.drop(drop_cols, axis=1, inplace=True)

print(f"Final shape        : {df.shape}")
print(f"Final columns      : {list(df.columns)}")

Duplicates removed : 203
Final shape        : (130553, 8)
Final columns      : ['Bill.NO', 'Date', 'Product Name', 'Qty', 'Rate', 'Amount', 'Total', 'IsReturn']


# 6. Outliers Check

In [ ]:
# We document outliers but DO NOT remove them
# High Qty = bulk purchases, High Rate = premium products — both are valid

df_sales = df[df['IsReturn'] == False].copy()

for col in ['Qty', 'Rate', 'Total']:
    Q1  = df_sales[col].quantile(0.25)
    Q3  = df_sales[col].quantile(0.75)
    IQR = Q3 - Q1
    out = df_sales[(df_sales[col] < Q1-1.5*IQR) |
                   (df_sales[col] > Q3+1.5*IQR)]

    print(f"{col:<8}: {len(out)} outliers ({len(out)/len(df_sales)*100:.1f}%) | max={df_sales[col].max()}")

# 7. Save Cleaned Dataset

In [ ]:
#df.to_csv("royal_store_cleaned", index=False)
print("file saved successfully!")